# Lab 06 — Confusion Matrices & Error Analysis

*AI Cybersecurity Masterclass · Lesson 06 · DSJ THE CRITTERS*

## Research question
Where does the model fail, and which errors create real security risk?

Lab 05 asked which model scored best. This lab asks a harder question: when the model is wrong, *how* is it wrong, and what would that cost a security team? A confusion matrix is an error map — the diagonal is what went right, and the off-diagonal cells are the story.

## Learning objectives
- read a binary confusion matrix as four security outcomes, not four numbers;
- separate false positives (analyst noise) from false negatives (hidden risk);
- build a multiclass confusion matrix to see which attack families get confused;
- interpret per-class precision, recall, and F1 — and treat low support with suspicion;
- pull the actual misclassified rows out of the test set and inspect them;
- turn each error pattern into a written observation, interpretation, and next test.

In [ ]:
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

# OPTIONAL — GOOGLE COLAB ONLY: clone the repo so src/ actually exists here
try:
    import google.colab
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()

# If running from the notebooks/ folder locally:
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
# OPTIONAL — GOOGLE COLAB ONLY
# Run this cell if the UNSW-NB15 CSV files are not already in DATA_DIR.

try:
    from google.colab import files

    if not (DATA_DIR / "UNSW_NB15_training-set.csv").exists():
        print(
            "Upload UNSW_NB15_training-set.csv, "
            "UNSW_NB15_testing-set.csv, and optionally UNSW_NB15_features.csv"
        )
        uploaded = files.upload()
        DATA_DIR.mkdir(parents=True, exist_ok=True)

        for filename, content in uploaded.items():
            (DATA_DIR / filename).write_bytes(content)

except ImportError:
    print("Not running in Colab. Put the dataset CSV files in:", DATA_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.evaluation import binary_metrics, binary_metrics_frame, multiclass_report, error_rows

train, test = load_unsw(DATA_DIR)

# Keep the untouched test frame — we need attack_cat later to see WHICH attacks get missed.
test = test.reset_index(drop=True)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

y_test = y_test.reset_index(drop=True)

model = random_forest_pipeline(X_train).fit(X_train, y_train)
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

print("Test rows:", len(y_test))
print("Actual attacks:", int(y_test.sum()), "| predicted attacks:", int(y_pred.sum()))

## Step 1 — The binary confusion matrix

Rows are what actually happened. Columns are what the model predicted. Four outcomes:

| | Predicted Normal | Predicted Attack |
|---|---|---|
| **Actual Normal** | True Negative | **False Positive** — analyst noise |
| **Actual Attack** | **False Negative** — missed attack | True Positive |

Do not just look at the diagonal.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print("True Negative  (normal, called normal) :", tn)
print("False Positive (normal, called attack) :", fp, "  <- analyst noise")
print("False Negative (attack, called normal) :", fn, "  <- hidden risk")
print("True Positive  (attack, called attack) :", tp)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ConfusionMatrixDisplay(cm, display_labels=["Normal", "Attack"]).plot(
    cmap="Blues", ax=axes[0], colorbar=False, values_format="d"
)
axes[0].set_title("Counts")

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred, labels=[0, 1], normalize="true"),
    display_labels=["Normal", "Attack"],
).plot(cmap="Blues", ax=axes[1], colorbar=False, values_format=".3f")
axes[1].set_title("Normalized by actual class (row-wise recall)")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "lab06_binary_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 2 — Put a cost on each error type

The same matrix, expressed as the two rates a security team actually argues about:

- **False positive rate** — of all genuinely normal traffic, how much did we alert on? Drives alert fatigue.
- **False negative rate** — of all real attacks, how many did we miss? Drives silent failure.

In [ ]:
metrics = binary_metrics(y_test, y_pred, y_score)
display(binary_metrics_frame("Random Forest", metrics).round(4))

print(f"Of {tn + fp} normal events, {fp} produced a false alarm  "
      f"({metrics.false_positive_rate:.2%} false positive rate)")
print(f"Of {fn + tp} real attacks, {fn} were missed entirely     "
      f"({metrics.false_negative_rate:.2%} false negative rate)")

## Step 3 — Multiclass confusion matrix by attack category

Binary tells you *that* attacks were missed. Multiclass tells you *which kinds*. Confusion between two attack families is often tolerable; confusing an attack with normal traffic is not.

The matrix is normalized by row, so each cell reads as "what fraction of this actual category was predicted as that."

In [ ]:
Xm_train, ym_train = split_xy(train, "attack_cat")
Xm_test, ym_test = split_xy(test, "attack_cat")
Xm_train, Xm_test = align_columns(Xm_train, Xm_test)

Xm_train = drop_identifier_like_columns(Xm_train)
Xm_test = drop_identifier_like_columns(Xm_test)
ym_test = ym_test.reset_index(drop=True)

multi_model = random_forest_pipeline(Xm_train).fit(Xm_train, ym_train)
ym_pred = multi_model.predict(Xm_test)

classes = sorted(set(ym_test) | set(ym_pred))

fig, ax = plt.subplots(figsize=(9.5, 8))
ConfusionMatrixDisplay(
    confusion_matrix(ym_test, ym_pred, labels=classes, normalize="true"),
    display_labels=classes,
).plot(cmap="Blues", ax=ax, colorbar=False, values_format=".2f", xticks_rotation=45)
ax.set_title("Attack category confusion (row-normalized)")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "lab06_multiclass_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 4 — Per-class report, read with support in mind

Support is how many real examples of that class exist in the test set. A class with 20 examples can post a flattering F1 that means almost nothing. Sort by recall to find who is being missed, then check whether support is large enough to trust the number.

In [ ]:
report = multiclass_report(ym_test, ym_pred)
summary = report.drop(index=["accuracy", "macro avg", "weighted avg"], errors="ignore")
summary = summary.sort_values("recall")

display(summary[["precision", "recall", "f1-score", "support"]].round(3))

print("Macro avg F1   :", round(report.loc["macro avg", "f1-score"], 4),
      "  <- treats every class equally")
print("Weighted avg F1:", round(report.loc["weighted avg", "f1-score"], 4),
      "  <- dominated by the common classes")

low_support = summary[summary["support"] < summary["support"].median()]
print("\nClasses with below-median support (interpret their scores cautiously):")
print(", ".join(low_support.index) if len(low_support) else "none")

## Step 5 — Extract the actual misclassified rows

Scores summarize. Rows explain. `error_rows` labels every test row as `correct`, `false_positive`, or `false_negative` so you can look at the evidence directly.

In [ ]:
errors = error_rows(X_test, y_test, y_pred)
print(errors["error_type"].value_counts().to_string())

false_negatives = errors[errors["error_type"] == "false_negative"]
false_positives = errors[errors["error_type"] == "false_positive"]

print("\nMissed attacks (false negatives):", false_negatives.shape)
print("False alarms   (false positives):", false_positives.shape)

false_negatives.to_csv(RESULTS_DIR / "lab06_false_negatives.csv", index=False)
false_positives.to_csv(RESULTS_DIR / "lab06_false_positives.csv", index=False)

display(false_negatives.head())

## Step 6 — Compare missed attacks against caught attacks

A list of missed rows is not yet a finding. Compare the two groups: if the missed attacks are systematically shorter, smaller, or use a different service, that is a feature hypothesis you can test.

In [ ]:
caught = errors[(errors["actual"] == 1) & (errors["error_type"] == "correct")]

numeric_cols = [c for c in ["dur", "sbytes", "dbytes", "spkts", "dpkts", "rate"]
                if c in errors.columns]

side_by_side = pd.DataFrame({
    "missed (FN) median": false_negatives[numeric_cols].median(),
    "caught (TP) median": caught[numeric_cols].median(),
})
side_by_side["ratio missed/caught"] = (
    side_by_side["missed (FN) median"] / side_by_side["caught (TP) median"].replace(0, np.nan)
)
display(side_by_side.round(3))

# Which attack families hide inside the false negatives?
missed_by_category = test.loc[false_negatives.index, "attack_cat"].value_counts()
attacks_by_category = test[test["label"] == 1]["attack_cat"].value_counts()

miss_rate = pd.DataFrame({
    "missed": missed_by_category,
    "total attacks": attacks_by_category,
}).dropna()
miss_rate["miss rate"] = (miss_rate["missed"] / miss_rate["total attacks"]).round(3)

print("\nMiss rate by attack category (worst first):")
display(miss_rate.sort_values("miss rate", ascending=False))

## Step 7 — Write three research notes

One sentence each, per finding. This is the part that separates a notebook from a research artifact.

- **Observation** — what did the number or figure actually show?
- **Interpretation** — why might that be happening? (low support, feature overlap, label limits, threshold placement)
- **Next test** — what single change would you make, and how would you measure whether it helped?

Replace the placeholder text below with your own, then run the cell to save it alongside your figures.

In [ ]:
notes = """# Lab 06 — Error Analysis Notes

## Note 1 — Missed attacks
Observation:
Interpretation:
Next test:

## Note 2 — False alarms
Observation:
Interpretation:
Next test:

## Note 3 — Weakest attack category
Observation:
Interpretation:
Next test:

## The one improvement I will test next
Change:
How I will measure it (same split, same metrics):
"""

notes_path = RESULTS_DIR / "lab06_error_analysis_notes.md"
notes_path.write_text(notes)

print("Saved:", notes_path)
for f in sorted(RESULTS_DIR.glob("lab06_*")):
    print("  -", f.name)

## Your analysis

Answer these before moving to Lesson 07:

1. Which error type is more expensive in your scenario — the false positives or the false negatives? Defend the answer operationally, not mathematically.
2. Which attack category has the worst miss rate, and is its support large enough to trust that number?
3. Look at the multiclass matrix: name one pair of categories the model confuses with each other. Is that confusion operationally acceptable?
4. Do the missed attacks differ measurably from the caught attacks in the table from Step 6? If yes, name the feature and the direction.
5. Every model in Lab 05 struggled on some of the same classes. Does that point to the algorithm, the features, or the labels — and how would you tell the difference?

**Deliverables:** binary confusion matrix, multiclass confusion matrix, per-class report, extracted false positives and false negatives, three written research notes, and one improvement chosen to test next.